# Phase 6 — Part 3D: Post-picking seismic attributes and Remaining Fluid Index

**Independent of Part 3C execution.** Read the exported horizon TXT files, Part 1 matched baseline/monitor SEG-Y, Part 3B sweetness cubes, Phase 4 fluid classifications, and well trajectories. **No horizon picking, no RMS cube cache, no Part 3A dependency.**

**Definitions:** RMS of paired finite baseline/monitor seismic samples in ±10 ms of each picked TWT; `dRMS = RMS_B − RMS_M`; `NdRMS = 200*dRMS/(RMS_B+RMS_M)`; `NRMS = 200*RMS(M−B)/(RMS_B+RMS_M)`. Sweetness comes from 3B and is sampled at horizon TWT. RFI is a configurable *relative screening score*, not a saturation or reserve estimate.

**Run Cells 1–7 sequentially.** Cell 2 verifies source files and horizon mapping before expensive extraction. Output is under `/data/epic20/APGCE/phase6_3D/`, leaving existing 3C exports untouched.


In [1]:
# CELL 1 — Imports and editable settings
from pathlib import Path
import os, re, json, math, warnings
import numpy as np
import pandas as pd
import segyio
import matplotlib.pyplot as plt
from matplotlib.path import Path as MplPath
from scipy.spatial import cKDTree
from scipy.ndimage import gaussian_filter
from IPython.display import display, clear_output

ROOT = Path('/data/epic20/APGCE')
HORIZON_DIR = ROOT/'phase6_horizons'
HORIZON_SUMMARY = HORIZON_DIR/'Autopick_Export_Summary.csv'
BASELINE_SGY = Path('/home/apgce_team_08/Phase6_Checkpoints/Baseline_Matched.sgy')
MONITOR_SGY = Path('/home/apgce_team_08/Phase6_Checkpoints/Monitor_Matched.sgy')
SWEET_DIR = ROOT/'phase6_attributes/Attributes_10ms_50pct_overlap'
WELL_DIR = Path('/home/apgce_team_08/To_Overlay_Horizon')
FLUID_CSV = Path('/home/apgce_team_08/link-to-apgce_team_08/Phase4_7Wells_Reservoir_Classification_TWT.csv')
PLATFORM_CSV = Path('/home/apgce_team_08/Platform.csv')
OUT = ROOT/'phase6_3D'
ATTR_OUT = OUT/'Horizon_Attributes'
RFI_OUT = OUT/'RFI'
for p in (ATTR_OUT,RFI_OUT):p.mkdir(parents=True,exist_ok=True)
HALF_WINDOW_MS = 10.0
TRACE_BLOCK = 512       # peak memory roughly 2*TRACE_BLOCK*nsamples float32
SWEET_INTERPOLATE = True
EPS = 1e-10
MAX_PLOT_POINTS = 110_000
RANDOM_SEED = 42
# Screening weights: high absolute 4D response reduces remaining potential.
WEIGHTS = {'base_rms':.05,'monitor_rms':.08,'base_sweet':.07,
           'monitor_sweet':.12,'ndrms':.38,'nrms':.18,'sweet_change':.12}
OIL_XY_RADIUS_M = 750.0
OIL_TWT_RADIUS_MS = 35.0
MAX_OIL_4D_PENALTY = 65.0
OUTSIDE_CLOSURE_RFI_CAP = 8.0
CONTOUR_INTERVAL_MS = 10.0
MIN_CLOSURE_RELIEF_MS = 5.0
MIN_CLOSURE_AREA_M2 = 10000.0
CLOSURE_SMOOTH_BINS = 2.0
PLATFORM_MODE = 'unconstrained' # unconstrained | all_platforms | single_platform
PLATFORM_INDEX = 0
PLATFORM_RADIUS_M = 750.0
PLOT_FONT_SIZE = 13
PLOT_FIGSIZE = (22,14)
PLOT_DPI = 180
# Missing trajectories are not silently replaced by guessed wellhead positions.
REQUIRE_OIL_WELLS = True
print('Cell 1 completed. Outputs:',OUT)


Cell 1 completed. Outputs: /data/epic20/APGCE/phase6_3D


In [2]:
# CELL 2 — Discover exported horizons and validate input geometry/time axes
if not HORIZON_SUMMARY.is_file():
    raise FileNotFoundError(f'Missing Cell 12 horizon export summary: {HORIZON_SUMMARY}')
summary = pd.read_csv(HORIZON_SUMMARY)
if not {'Horizon','File'}.issubset(summary):
    raise ValueError(f'Horizon summary requires Horizon and File; found {list(summary)}')
horizons = {}
for row in summary.itertuples(index=False):
    name = str(row.Horizon)
    path = HORIZON_DIR/str(row.File)
    if not path.is_file():raise FileNotFoundError(f'Horizon {name}: {path}')
    if name in horizons:raise ValueError(f'Duplicate horizon name {name}')
    horizons[name]=path
if not horizons:raise ValueError('No exported horizons in summary')
required_cols = ['X','Y','Inline','Crossline','TWT']
for name,path in horizons.items():
    head = pd.read_csv(path,sep='\t',nrows=3)
    if not set(required_cols).issubset(head):
        raise ValueError(f'{path}: missing {set(required_cols)-set(head)}')
for path in [BASELINE_SGY,MONITOR_SGY,SWEET_DIR/'sweetness_metadata.json'] + [SWEET_DIR/(x+'.npy') for x in ['Sweetness_Baseline','Sweetness_Monitor','Sweetness_Difference']]:
    if not path.is_file():raise FileNotFoundError(f'Missing required input: {path}')
with segyio.open(str(BASELINE_SGY),'r',ignore_geometry=True) as b, segyio.open(str(MONITOR_SGY),'r',ignore_geometry=True) as m:
    if b.tracecount != m.tracecount or len(b.samples)!=len(m.samples):
        raise ValueError('Baseline/monitor trace counts or sample counts differ')
    axis_b=np.asarray(b.samples,dtype=np.float64)
    axis_m=np.asarray(m.samples,dtype=np.float64)
    if not np.allclose(axis_b,axis_m,atol=1e-5,rtol=0):raise ValueError('Baseline/monitor TWT axes differ')
    if np.any(np.diff(axis_b)<=0):raise ValueError('Non-monotonic seismic TWT axis')
    dt_ms=float(np.median(np.diff(axis_b)))
    if not np.allclose(np.diff(axis_b),dt_ms,atol=1e-4,rtol=1e-5):raise ValueError('Nonuniform seismic sample interval')
    ntr=b.tracecount
    # Inline/crossline from SEG-Y header, independent of cache or 3C kernel.
    il_b=np.asarray(b.attributes(segyio.TraceField.INLINE_3D)[:],dtype=np.int64)
    xl_b=np.asarray(b.attributes(segyio.TraceField.CROSSLINE_3D)[:],dtype=np.int64)
    il_m=np.asarray(m.attributes(segyio.TraceField.INLINE_3D)[:],dtype=np.int64)
    xl_m=np.asarray(m.attributes(segyio.TraceField.CROSSLINE_3D)[:],dtype=np.int64)
    if not np.array_equal(il_b,il_m) or not np.array_equal(xl_b,xl_m):
        raise ValueError('Part 1 SEG-Y baseline/monitor inline/crossline trace-order mismatch')
    geom=pd.DataFrame({'Inline':il_b,'Crossline':xl_b,'Trace_Index':np.arange(ntr,dtype=np.int64)})
    if geom.duplicated(['Inline','Crossline']).any():raise ValueError('Duplicate seismic bins: ambiguous horizon mapping')
    trace_lookup=geom.set_index(['Inline','Crossline'])['Trace_Index']
    # QC several traces, not only metadata.
    for idx in np.unique(np.linspace(0,ntr-1,min(17,ntr),dtype=int)):
        if int(b.header[int(idx)][segyio.TraceField.INLINE_3D]) != int(m.header[int(idx)][segyio.TraceField.INLINE_3D]):
            raise ValueError('Trace header mismatch')
meta=json.loads((SWEET_DIR/'sweetness_metadata.json').read_text())
centers=np.asarray(meta['centers_ms'],dtype=float)
if len(centers)<2 or np.any(np.diff(centers)<=0):raise ValueError('Invalid Part 3B sweetness centers_ms')
sweet={k:np.load(SWEET_DIR/(k+'.npy'),mmap_mode='r') for k in ['Sweetness_Baseline','Sweetness_Monitor','Sweetness_Difference']}
for k,v in sweet.items():
    if v.shape!=(ntr,len(centers)):
        raise ValueError(f'{k} shape {v.shape} != ({ntr},{len(centers)})')
print('Horizon files:',len(horizons),'| seismic traces:',f'{ntr:,}','| samples:',len(axis_b),'| dt:',dt_ms,'ms')
print('Sweetness centers:',len(centers),'range:',(centers[0],centers[-1]))
print('Cell 2 completed: independent SEG-Y and sweetness validation passed')


Horizon files: 7 | seismic traces: 1,240,941 | samples: 1301 | dt: 2.0 ms
Sweetness centers: 259 range: (np.float64(10.0), np.float64(2590.0))
Cell 2 completed: independent SEG-Y and sweetness validation passed


In [3]:
# CELL 3 — Direct seismic RMS ±10ms on each picked horizon; sample 3B sweetness
# One SEG-Y open per horizon; read only the seismic traces that are needed, in bounded blocks.
def rms_pair_windows(bdata,mdata,lo,hi):
    n=len(lo); rb=np.full(n,np.nan);rm=np.full(n,np.nan);rd=np.full(n,np.nan)
    for j in range(n):
        if lo[j]<0 or hi[j]<=lo[j]:continue
        a=np.asarray(bdata[j,lo[j]:hi[j]],dtype=np.float64)
        c=np.asarray(mdata[j,lo[j]:hi[j]],dtype=np.float64)
        good=np.isfinite(a)&np.isfinite(c)
        if not good.any():continue
        a=a[good];c=c[good]
        rb[j]=np.sqrt(np.mean(a*a));rm[j]=np.sqrt(np.mean(c*c))
        delta=c-a
        rd[j]=np.sqrt(np.mean(delta*delta))
    return rb,rm,rd

def sample_sweet(cube,ids,twt):
    result=np.full(len(ids),np.nan,dtype=np.float64)
    ok=(ids>=0)&np.isfinite(twt)&(twt>=centers[0])&(twt<=centers[-1])
    if not ok.any():return result
    idx=np.flatnonzero(ok); times=twt[idx]
    right=np.searchsorted(centers,times,side='left')
    right=np.clip(right,0,len(centers)-1)
    left=np.maximum(0,right-1)
    if SWEET_INTERPOLATE:
        a=np.asarray(cube[ids[idx],left],float)
        b=np.asarray(cube[ids[idx],right],float)
        w=np.divide(times-centers[left],centers[right]-centers[left],
                    out=np.zeros(len(idx)),where=right!=left)
        exact=(right==left)|(np.abs(times-centers[right])<1e-8)
        # Do not bridge invalid sweetness samples except at an exact center.
        both=np.isfinite(a)&np.isfinite(b)
        result[idx[both]]=a[both]*(1-w[both])+b[both]*w[both]
        result[idx[exact & np.isfinite(b)]]=b[exact & np.isfinite(b)]
    else:
        nearest=np.where(np.abs(times-centers[left])<=np.abs(times-centers[right]),left,right)
        result[idx]=np.asarray(cube[ids[idx],nearest],float)
    return result

def extract_horizon(name,path):
    d=pd.read_csv(path,sep='\t',usecols=required_cols)
    for c in required_cols:d[c]=pd.to_numeric(d[c],errors='coerce')
    if d[['Inline','Crossline']].isna().any().any():raise ValueError(f'{name}: invalid IL/XL')
    if d.duplicated(['Inline','Crossline']).any():raise ValueError(f'{name}: duplicate horizon bins')
    # Use exact bin keys, not row order or nearest XY; no silent trace substitution.
    keys=pd.MultiIndex.from_frame(d[['Inline','Crossline']].astype(np.int64))
    ids=trace_lookup.reindex(keys).to_numpy(dtype=float)
    if np.isnan(ids).any():raise ValueError(f'{name}: {np.isnan(ids).sum()} horizon bins missing from Part 1 SEG-Y')
    ids=ids.astype(np.int64)
    t=d['TWT'].to_numpy(float)
    lo=np.searchsorted(axis_b,t-HALF_WINDOW_MS,side='left')
    hi=np.searchsorted(axis_b,t+HALF_WINDOW_MS,side='right')
    inside=np.isfinite(t)&(t-HALF_WINDOW_MS>=axis_b[0])&(t+HALF_WINDOW_MS<=axis_b[-1])
    lo[~inside]=-1;hi[~inside]=-1
    rb=np.full(len(d),np.nan);rm=rb.copy();rd=rb.copy()
    with segyio.open(str(BASELINE_SGY),'r',ignore_geometry=True) as bf, segyio.open(str(MONITOR_SGY),'r',ignore_geometry=True) as mf:
        for start in range(0,len(d),TRACE_BLOCK):
            end=min(start+TRACE_BLOCK,len(d))
            # Load only the requested trace IDs, not entire volumes.
            bb=np.stack([np.asarray(bf.trace[int(i)],dtype=np.float32) for i in ids[start:end]])
            mm=np.stack([np.asarray(mf.trace[int(i)],dtype=np.float32) for i in ids[start:end]])
            rb[start:end],rm[start:end],rd[start:end]=rms_pair_windows(bb,mm,lo[start:end],hi[start:end])

    den=rb+rm
    good=np.isfinite(rb)&np.isfinite(rm)&np.isfinite(rd)&(den>EPS)
    dr=np.full(len(d),np.nan);nd=np.full(len(d),np.nan);nr=np.full(len(d),np.nan)
    dr[good]=rb[good]-rm[good]
    nd[good]=200*dr[good]/den[good]
    nr[good]=200*rd[good]/den[good]
    d['RMS_Baseline']=rb;d['RMS_Monitor']=rm;d['RMS_Difference']=rd
    d['dRMS']=dr;d['NdRMS_pct']=nd;d['NRMS']=nr
    for k,v in sweet.items():d[k]=sample_sweet(v,ids,t)
    return d

attr_files={}
for index,(name,path) in enumerate(horizons.items(),1):
    safe=re.sub(r'[^A-Za-z0-9_.-]+','_',name)
    dest=ATTR_OUT/(safe+'_3D_Attributes.csv')
    d=extract_horizon(name,path)
    # Export 12 significant figures for robust formula readback.
    tmp=dest.with_suffix('.partial.csv')
    d.to_csv(tmp,index=False,float_format='%.12g',na_rep='NaN')
    tmp.replace(dest)
    attr_files[name]=dest
    print(f'[{index}/{len(horizons)}] {name}: {len(d):,} picks, {d.NdRMS_pct.notna().sum():,} valid RMS pairs, {d.Sweetness_Baseline.notna().sum():,} sweetness')
print('Cell 3 completed. Attributes:',ATTR_OUT)


[1/7] I-25: 1,240,906 picks, 1,240,906 valid RMS pairs, 1,240,906 sweetness
[2/7] I-35: 1,240,921 picks, 1,240,921 valid RMS pairs, 1,240,921 sweetness
[3/7] I-35_LOWER: 1,240,877 picks, 1,240,877 valid RMS pairs, 1,240,877 sweetness
[4/7] I-15.0L: 1,240,875 picks, 1,240,875 valid RMS pairs, 1,240,875 sweetness
[5/7] I-68: 1,240,696 picks, 1,240,696 valid RMS pairs, 1,240,696 sweetness
[6/7] I-85: 1,240,904 picks, 1,240,904 valid RMS pairs, 1,240,904 sweetness
[7/7] I-100: 1,240,825 picks, 1,240,825 valid RMS pairs, 1,240,825 sweetness
Cell 3 completed. Attributes: /data/epic20/APGCE/phase6_3D/Horizon_Attributes


In [4]:
# CELL 4 — Independent QC of saved attributes; abort on physically inconsistent calculations
qc=[]
for name,path in attr_files.items():
    d=pd.read_csv(path)
    b=d.RMS_Baseline.to_numpy(float);m=d.RMS_Monitor.to_numpy(float)
    diff=d.RMS_Difference.to_numpy(float);dr=d.dRMS.to_numpy(float)
    nd=d.NdRMS_pct.to_numpy(float);nr=d.NRMS.to_numpy(float)
    valid=np.isfinite(b)&np.isfinite(m)&np.isfinite(diff)&np.isfinite(dr)&np.isfinite(nd)&np.isfinite(nr)&((b+m)>EPS)
    if not valid.any():raise AssertionError(f'{name}: no valid RMS pairs')
    # Relative tolerance accounts for 12-significant-digit CSV readback and RMS magnitudes.
    err_d=np.abs(dr[valid]-(b[valid]-m[valid]))
    err_n=np.abs(nd[valid]-200*(b[valid]-m[valid])/(b[valid]+m[valid]))
    err_r=np.abs(nr[valid]-200*diff[valid]/(b[valid]+m[valid]))
    mis=int(np.count_nonzero(((dr[valid]>0)&(nd[valid]<0))|((dr[valid]<0)&(nd[valid]>0))))
    d_ok=np.allclose(dr[valid],b[valid]-m[valid],rtol=2e-9,atol=1e-5)
    n_ok=np.allclose(nd[valid],200*(b[valid]-m[valid])/(b[valid]+m[valid]),rtol=2e-9,atol=1e-5)
    r_ok=np.allclose(nr[valid],200*diff[valid]/(b[valid]+m[valid]),rtol=2e-9,atol=1e-5)
    qc.append({'Horizon':name,'Valid':int(valid.sum()),'Sign_mismatches':mis,
               'Max_dRMS_error':err_d.max(),'Max_NdRMS_error_pct':err_n.max(),
               'Max_NRMS_error_pct':err_r.max(),'Pass':bool(mis==0 and d_ok and n_ok and r_ok)})
qc_table=pd.DataFrame(qc)
display(qc_table)
qc_table.to_csv(OUT/'Horizon_QC.csv',index=False)
if not qc_table.Pass.all():raise AssertionError('3D horizon RMS QC failed. Inspect Horizon_QC.csv before RFI.')
print('Cell 4 completed: all horizon attribute calculations pass QC')


,Horizon,Valid,Sign_mismatches,Max_dRMS_error,Max_NdRMS_error_pct,Max_NRMS_error_pct,Pass
0,I-25,1240906,0,1.000062e-07,9.596135e-10,1.334769e-09,True
1,I-35,1240921,0,1.000064e-07,9.762409e-10,1.513911e-09,True
2,I-35_LOWER,1240877,0,1.000080e-07,9.526836e-10,1.403293e-09,True
3,I-15.0L,1240875,0,1.000080e-07,9.726110e-10,1.384194e-09,True
4,I-68,1240696,0,1.000080e-07,9.491279e-10,1.236515e-09,True
5,I-85,1240904,0,1.000044e-07,9.756103e-10,1.296542e-09,True
6,I-100,1240825,0,1.000080e-07,9.607901e-10,1.241602e-09,True


Cell 4 completed: all horizon attribute calculations pass QC


In [5]:
# CELL 5 — Well trajectories (checkshot + deviation; vertical if deviation missing)
# NOTEBOOK CELL 10
# WELL OVERLAY HELPER — run once before Cells 23, 11 and 29.
# Requires numpy, pandas, scipy. Does not modify seismic or horizon arrays.
from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from functools import lru_cache

WELL_OVERLAY_FOLDER = Path("/home/apgce_team_08/To_Overlay_Horizon/")
# LAS X/Y header fields are the authoritative wellhead coordinates.
# Copy the eight Well-*.las files into WELL_OVERLAY_FOLDER.
WELL_OVERLAY_KB_M = {
    "Well_1":9.450, "Well_2":22.000, "Well_3":22.000,
    "Well_4":21.978, "Well_5":22.000, "Well_6":22.000,
    "Well_7":29.000, "Well_8":29.800}
WELL_MARKER_SIZE = 95        # EDIT HERE: scatter size
WELL_SHOW_LABELS = True      # EDIT HERE
WELL_SECTION_TOL_BINS = 2.0  # EDIT HERE: section-line proximity
WELL_MAP_MAX_OFFSET_M = 200. # EDIT HERE: maximum nearest survey-bin distance
WELL_TRAJECTORY_STEP_M = 5.0 # EDIT HERE: trajectory interpolation interval
WELL_ALLOW_CHECKSHOT_EXTRAPOLATION = False # EDIT HERE; default: avoid extrapolation

def _well_file(w, kind):
    # Matches Well-4_Checkshot.txt as well as Well-3_Checkshot.las.
    opts = list(WELL_OVERLAY_FOLDER.glob(f"Well-{w}_{kind}*"))
    if not opts:
        opts = list(WELL_OVERLAY_FOLDER.glob(f"Well_{w}_{kind}*"))
    return sorted(opts)[0] if opts else None

def _numeric_file(path, las=False):
    if las:
        lines=path.read_text(errors="replace").splitlines()
        start=next((i for i,l in enumerate(lines) if l.strip().upper().startswith("~ASCII")),None)
        if start is None: raise ValueError(f"Missing LAS ~ASCII section: {path}")
        lines=lines[start+1:]
    else:
        lines=path.read_text(errors="replace").splitlines()
    vals=[]
    for line in lines:
        if not line.strip() or line.lstrip().startswith(("#","~")): continue
        try:
            row=[float(x) for x in line.replace(","," ").split()]
            if len(row)>=2 and np.isfinite(row[:2]).all() and not any(v==-999.25 for v in row):
                vals.append(row)
        except ValueError:
            continue
    if not vals: raise ValueError(f"No numeric rows in {path}")
    width=min(map(len,vals))
    return np.array([r[:width] for r in vals],float)

def _checkshot(w):
    path=_well_file(w,"Checkshot")
    if path is None: return None
    a=_numeric_file(path,las=path.suffix.lower()==".las")
    # Checkshot columns are TVDSS (m), TWT (ms), verified from uploaded files.
    a=a[np.argsort(a[:,0])]
    a=a[np.r_[True,np.diff(a[:,0])>0]]
    return a[:,:2]

def _deviation(w):
    path=_well_file(w,"Deviation")
    if path is None: return None
    a=_numeric_file(path)
    if a.shape[1]<4: raise ValueError(f"Deviation must contain MDKB,TVD,INCL,AZIM: {path}")
    a=a[np.argsort(a[:,0])]
    a=a[np.r_[True,np.diff(a[:,0])>0]]
    return a[:,:4]

def _anchor_records():
    records={}
    for w in range(1,9):
        files=sorted(WELL_OVERLAY_FOLDER.glob(f"Well-{w}.las"))
        if not files:
            files=sorted(WELL_OVERLAY_FOLDER.glob(f"Well-{w}(*).las"))
        if not files:
            print(f"Well_{w}: LAS wellhead file missing")
            continue
        # Read only LAS header, not the log curve data.
        coords={}
        for line in files[0].read_text(errors="replace").splitlines():
            if line.lstrip().upper().startswith(("~A","~ASCII")): break
            m=re.match(r"^\s*([XY])\s*\.\s*\S*\s+([-+]?\d+(?:\.\d+)?)\s*(?::|$)",line,re.I)
            if m: coords[m.group(1).upper()]=float(m.group(2))
        if set(coords)!={"X","Y"}:
            print(f"Well_{w}: missing X/Y LAS header in {files[0].name}")
            continue
        key=f"Well_{w}"
        records[key]=[(coords["X"],coords["Y"],-WELL_OVERLAY_KB_M[key],"tvdss")]
        print(f"{key}: LAS wellhead X={coords['X']:.2f}, Y={coords['Y']:.2f}")
    return records

def load_well_overlay_trajectories():
    anchors=_anchor_records()
    tracks={}
    warnings=[]
    for w in range(1,9):
        key=f"Well_{w}"
        cs=_checkshot(w)
        if cs is None:
            warnings.append(f"{key}: missing checkshot; omitted")
            continue
        dev=_deviation(w)
        kb=WELL_OVERLAY_KB_M[key]
        lo,hi=cs[0,0],cs[-1,0]
        depths=np.arange(lo,hi+WELL_TRAJECTORY_STEP_M,WELL_TRAJECTORY_STEP_M)
        depths=np.unique(np.r_[depths[depths<=hi],hi])
        twt=np.interp(depths,cs[:,0],cs[:,1])
        if dev is None:
            east=np.zeros(len(depths));north=np.zeros(len(depths))
            kind="vertical (no deviation)"
        else:
            # Deviation: MDKB, TVD (below KB), inclination, azimuth clockwise from North.
            # Minimum curvature, interpolated in MD and mapped onto TVDSS=TVD-KB.
            md,tvd,inc,azi=dev.T
            inc=np.deg2rad(inc);azi=np.deg2rad(azi)
            e=np.zeros(len(md));n=np.zeros(len(md))
            for k in range(1,len(md)):
                dog=np.arccos(np.clip(np.cos(inc[k-1])*np.cos(inc[k])+
                     np.sin(inc[k-1])*np.sin(inc[k])*np.cos(azi[k]-azi[k-1]),-1,1))
                rf=1. if dog<1e-9 else 2*np.tan(dog/2)/dog
                dm=md[k]-md[k-1]
                e[k]=e[k-1]+dm/2*(np.sin(inc[k-1])*np.sin(azi[k-1])+
                                  np.sin(inc[k])*np.sin(azi[k]))*rf
                n[k]=n[k-1]+dm/2*(np.sin(inc[k-1])*np.cos(azi[k-1])+
                                  np.sin(inc[k])*np.cos(azi[k]))*rf
            tvdss=tvd-kb
            east=np.interp(depths,tvdss,e)
            north=np.interp(depths,tvdss,n)
            kind="deviated"
        well_anchors=anchors.get(key,[])
        if not well_anchors:
            warnings.append(f"{key}: no X/Y positional anchor; omitted (LAS X/Y missing)")
            continue
        # LAS X/Y are wellhead positions at TVDSS=-KB.
        origins=[]
        for ax,ay,reference,unit in well_anchors:
            depth_ref=np.interp(reference,cs[:,1],cs[:,0]) if unit=="twt" else reference
            if dev is not None:
                tvdss=tvd-kb
                if not (tvdss[0]<=depth_ref<=tvdss[-1]): continue
                dx=np.interp(depth_ref,tvdss,e)
                dy=np.interp(depth_ref,tvdss,n)
            else: dx=dy=0.
            origins.append((ax-dx,ay-dy))
        if not origins:
            warnings.append(f"{key}: no usable positional anchors; omitted")
            continue
        origin=np.median(np.asarray(origins),axis=0)
        spread=float(np.max(np.linalg.norm(np.asarray(origins)-origin,axis=1)))
        if spread>WELL_MAP_MAX_OFFSET_M:
            warnings.append(f"{key}: conflicting anchor positions (max residual {spread:.1f} m)")
        tracks[key]=pd.DataFrame({"X":origin[0]+east,"Y":origin[1]+north,
                                  "TWT":twt,"TVDSS":depths})
        print(f"{key}: {kind}; {len(well_anchors)} positional anchors; "
              f"checkshot {twt.min():.1f}-{twt.max():.1f} ms")
    for warning in warnings: print("Well overlay QC:",warning)
    return tracks

WELL_OVERLAY_TRACKS=load_well_overlay_trajectories()


# Only 7 wells are used for oil-fluid constraints; Well 3 is excluded.
WELL_OVERLAY_TRACKS={k:v for k,v in WELL_OVERLAY_TRACKS.items() if k != 'Well_3'}
print('Cell 5 completed:',len(WELL_OVERLAY_TRACKS),'usable trajectories')


Well_1: LAS wellhead X=470216.40, Y=577913.78
Well_2: LAS wellhead X=471455.52, Y=573417.89
Well_3: LAS wellhead X=476760.33, Y=569521.16
Well_4: LAS wellhead X=470944.51, Y=575660.47
Well_5: LAS wellhead X=462919.18, Y=575759.86
Well_6: LAS wellhead X=463912.30, Y=570036.45
Well_7: LAS wellhead X=462597.31, Y=579774.76
Well_8: LAS wellhead X=469356.85, Y=570185.41
Well_1: vertical (no deviation); 1 positional anchors; checkshot 186.8-2436.0 ms
Well_2: deviated; 1 positional anchors; checkshot 441.4-2538.5 ms
Well_3: vertical (no deviation); 1 positional anchors; checkshot 1143.5-2539.3 ms
Well_4: deviated; 1 positional anchors; checkshot 1151.8-2158.4 ms
Well_5: deviated; 1 positional anchors; checkshot 1163.2-1937.8 ms
Well_6: deviated; 1 positional anchors; checkshot 564.4-1910.0 ms
Well_7: vertical (no deviation); 1 positional anchors; checkshot 1086.2-1860.0 ms
Well_8: vertical (no deviation); 1 positional anchors; checkshot 141.4-1768.2 ms
Cell 5 completed: 7 usable trajectories


In [6]:
# CELL 6 — Well-fluid samples and structural-closure helpers
def _clean_well(v):
    s=str(v).strip().upper().replace('_','-').replace(' ','')
    m=re.search(r'(?:WELL-?)?(\d+)$',s)
    return 'Well-'+str(int(m.group(1))) if m else s

def load_oil_twt_with_trajectory(path=FLUID_CSV):
    if not path.is_file(): raise FileNotFoundError(f'Missing Phase 4 fluid file: {path}')
    raw=pd.read_csv(path,encoding='utf-8-sig',low_memory=False)
    raw.columns=[str(x).strip() for x in raw.columns]
    aliases={'well':['Well','Well_Name','Well_ID'],
             'twt':['TWT_ms','TWT','Seismic_TWT_ms'],
             'fluid':['ML_FLUID_FLAG','Fluid_Class','Fluid_Code','ML_FLUID_CLASS']}
    def col(keys):
        d={re.sub('[^a-z0-9]','',c.lower()):c for c in raw.columns}
        return next((d[k] for v in keys if (k:=re.sub('[^a-z0-9]','',v.lower())) in d),None)
    found={k:col(v) for k,v in aliases.items()}
    if any(v is None for v in found.values()):
        raise ValueError(f'Phase 4 columns missing. Found {found}; available {list(raw.columns)}')
    fluid=raw[found['fluid']]
    cls=pd.to_numeric(fluid,errors='coerce')
    mask=cls.eq(2)|fluid.astype(str).str.strip().str.lower().isin(['oil','oil-bearing','oil bearing','oil_bearing'])
    oil=raw.loc[mask,[found['well'],found['twt']]].copy()
    oil.columns=['Well','TWT']
    oil['TWT']=pd.to_numeric(oil.TWT,errors='coerce')
    oil['Well']=oil.Well.map(_clean_well)
    oil=oil.dropna(subset=['TWT'])
    if not len(oil):raise ValueError('Phase 4 contains no valid class-2 oil TWT samples')
    if 'WELL_OVERLAY_TRACKS' not in globals() or not WELL_OVERLAY_TRACKS:
        raise RuntimeError('Well trajectories unavailable. Run the earlier WELL OVERLAY HELPER cell; it loads LAS, deviation and checkshot files.')
    tracks={_clean_well(k):v for k,v in WELL_OVERLAY_TRACKS.items()}
    output=[];report=[]
    for well,part in oil.groupby('Well'):
        track=tracks.get(well)
        if track is None:
            report.append(f'{well}: no trajectory; skipped {len(part)} oil samples')
            continue
        t=track[['TWT','X','Y']].apply(pd.to_numeric,errors='coerce').dropna().sort_values('TWT')
        t=t.drop_duplicates('TWT')
        if len(t)<2:
            report.append(f'{well}: insufficient trajectory TWT points')
            continue
        samples=part.TWT.to_numpy(float)
        inside=(samples>=t.TWT.iloc[0])&(samples<=t.TWT.iloc[-1])
        if (~inside).any():report.append(f'{well}: skipped {int((~inside).sum())} samples outside checkshot coverage')
        valid=samples[inside]
        if len(valid):
            output.append(pd.DataFrame({'Well':well,'TWT':valid,
                'X':np.interp(valid,t.TWT,t.X),'Y':np.interp(valid,t.TWT,t.Y)}))
    if not output:
        raise ValueError('No oil samples intersect available well trajectories. Check TWT units, well IDs and checkshot coverage.')
    result=pd.concat(output,ignore_index=True).drop_duplicates(['Well','TWT'])
    print('Oil samples positioned on well trajectories:',len(result),'from',result.Well.nunique(),'wells')
    for msg in report:print('QC:',msg)
    return result

phase4_oil=load_oil_twt_with_trajectory()

def _scaled_oil_support(df,oil,rxy,rtwt):
    if rxy<=0 or rtwt<=0:raise ValueError('Oil proximity radii must be positive')
    coords=oil[['X','Y','TWT']].to_numpy(float)/np.array([rxy,rxy,rtwt])
    tree=cKDTree(coords)
    query=df[['X','Y','TWT']].to_numpy(float)/np.array([rxy,rxy,rtwt])
    out=np.zeros(len(df));ok=np.isfinite(query).all(axis=1)
    if ok.any():
        dist=tree.query(query[ok],workers=-1)[0]
        out[ok]=np.clip(1-dist,0,1)**2
    return out

def _closure_mask(df, interval_ms=10., min_relief_ms=5., min_area_m2=10000.,
                  smoothing_bins=2., max_grid=800):
    """Detect CLOSED STRUCTURAL HIGHS on the picked TWT horizon.
    Closed TWT contours are accepted only when the interior is shallower
    than the surrounding contour and the polygon has adequate relief/area.
    Edge-touching or unclosed contours are excluded. No closure is inferred
    when the TWT surface cannot be gridded reliably.
    """
    if interval_ms<=0 or min_relief_ms<0 or min_area_m2<0:raise ValueError('Invalid closure parameters')
    il=np.sort(df.Inline.dropna().unique());xl=np.sort(df.Crossline.dropna().unique())
    if len(il)<3 or len(xl)<3:raise ValueError('Not enough grid dimensions for closure contours')
    ii=pd.Index(il).get_indexer(df.Inline);jj=pd.Index(xl).get_indexer(df.Crossline)
    good=(ii>=0)&(jj>=0)
    shape=(len(il),len(xl))
    z=np.full(shape,np.nan);x=np.full(shape,np.nan);y=np.full(shape,np.nan)
    z[ii[good],jj[good]]=df.TWT.to_numpy(float)[good]
    x[ii[good],jj[good]]=df.X.to_numpy(float)[good]
    y[ii[good],jj[good]]=df.Y.to_numpy(float)[good]
    # Limit expensive contour geometry on very large horizons.
    stride=max(1,int(np.ceil(max(shape)/max_grid)))
    zz=z[::stride,::stride];xx=x[::stride,::stride];yy=y[::stride,::stride]
    valid=np.isfinite(zz)&np.isfinite(xx)&np.isfinite(yy)
    if valid.sum()<30:raise ValueError('Insufficient finite XY/TWT cells for closure QC')
    # Smooth only supported data, do not invent contours across NaN gaps.
    if smoothing_bins>0:
        num=gaussian_filter(np.where(valid,zz,0),smoothing_bins)
        den=gaussian_filter(valid.astype(float),smoothing_bins)
        zz=np.where(valid & (den>0.5),num/np.maximum(den,1e-9),np.nan)
    levels=np.arange(np.ceil(np.nanmin(zz)/interval_ms)*interval_ms,
                     np.nanmax(zz),interval_ms)
    # Grid index space makes boundary/closure checks stable even if XY is rotated.
    gy,gx=np.indices(zz.shape)
    closure=np.zeros(zz.shape,bool)
    paths=[]
    if len(levels):
        fig,ax=plt.subplots()
        cs=ax.contour(gx,gy,np.ma.masked_invalid(zz),levels=levels)
        for level,segments in zip(cs.levels,cs.allsegs):
            for vertices in segments:
                if len(vertices)<5 or np.linalg.norm(vertices[0]-vertices[-1])>1.5:continue
                poly=MplPath(vertices,closed=True)
                minx,miny=vertices.min(axis=0);maxx,maxy=vertices.max(axis=0)
                if minx<=1 or miny<=1 or maxx>=zz.shape[1]-2 or maxy>=zz.shape[0]-2:continue
                i0=max(0,int(np.floor(miny)));i1=min(zz.shape[0],int(np.ceil(maxy))+1)
                j0=max(0,int(np.floor(minx)));j1=min(zz.shape[1],int(np.ceil(maxx))+1)
                local=np.column_stack([gx[i0:i1,j0:j1].ravel(),gy[i0:i1,j0:j1].ravel()])
                inside=poly.contains_points(local).reshape((i1-i0,j1-j0))
                inside_valid=inside & np.isfinite(zz[i0:i1,j0:j1])
                if inside_valid.sum()<5:continue
                # A shallower interior is a structural high in TWT.
                interior=np.nanmin(zz[i0:i1,j0:j1][inside_valid])
                if level-interior<min_relief_ms:continue
                local_x=xx[i0:i1,j0:j1][inside_valid];local_y=yy[i0:i1,j0:j1][inside_valid]
                if len(local_x)<5:continue
                # Estimate physical bin area from median adjacent coordinate steps.
                dx=np.nanmedian(np.hypot(np.diff(xx,axis=1),np.diff(yy,axis=1)))
                dy=np.nanmedian(np.hypot(np.diff(xx,axis=0),np.diff(yy,axis=0)))
                area=float(inside_valid.sum()*dx*dy)
                if not np.isfinite(area) or area<min_area_m2:continue
                # Reject closure polygons enclosing missing data: their shape is uncertain.
                if np.any(inside & ~np.isfinite(zz[i0:i1,j0:j1])):continue
                closure[i0:i1,j0:j1] |= inside_valid
                paths.append((float(level),vertices.copy(),area))
        plt.close(fig)
    # Map closure mask back to full seismic grid by nearest downsampled grid bin.
    out=np.zeros(len(df),bool)
    valid_rows=good & np.isfinite(df.TWT.to_numpy(float))
    if valid_rows.any():
        ri=np.minimum(ii[valid_rows]//stride,closure.shape[0]-1)
        rj=np.minimum(jj[valid_rows]//stride,closure.shape[1]-1)
        out[valid_rows]=closure[ri,rj]
    return out,paths
# The Phase 4 file must have class-2 oil samples with mapped trajectory X/Y.
phase4_oil=load_oil_twt_with_trajectory()
if REQUIRE_OIL_WELLS and phase4_oil.empty:
    raise ValueError('No oil samples projected onto well trajectories')
print('Cell 6 completed. Positioned oil samples:',len(phase4_oil))


Oil samples positioned on well trajectories: 263 from 4 wells
Oil samples positioned on well trajectories: 263 from 4 wells
Cell 6 completed. Positioned oil samples: 263


In [12]:
# CELL 7 — Standalone RFI scoring, 7-panel viewer and exports
import ipywidgets as widgets
from scipy.spatial import cKDTree
from scipy.interpolate import griddata

def percentile_score(x, high_is_good=True):
    a=np.asarray(x,dtype=float)
    good=np.isfinite(a)
    out=np.full(len(a),np.nan)
    if good.sum()<10:return out
    lo,hi=np.nanpercentile(a[good],[5,95])
    if not np.isfinite(lo+hi) or hi<=lo:return out
    out[good]=np.clip((a[good]-lo)/(hi-lo),0,1)
    if not high_is_good:out[good]=1-out[good]
    return out

def load_platforms():
    if not PLATFORM_CSV.is_file():return pd.DataFrame(columns=['Name','X','Y'])
    p=pd.read_csv(PLATFORM_CSV,encoding='utf-8-sig')
    if not {'X(m)','Y(m)'}.issubset(p):raise ValueError(f'Platform file needs X(m), Y(m): {PLATFORM_CSV}')
    namecol=next((x for x in ['Platform','Name','Platform_Name'] if x in p),None)
    return pd.DataFrame({'Name':p[namecol].astype(str) if namecol else ['Platform '+str(i+1) for i in range(len(p))],
                         'X':pd.to_numeric(p['X(m)'],errors='coerce'),
                         'Y':pd.to_numeric(p['Y(m)'],errors='coerce')}).dropna().reset_index(drop=True)
platforms=load_platforms()

def compute_rfi_3d(name, weights=WEIGHTS, oil_xy_radius=OIL_XY_RADIUS_M,
                   oil_twt_radius=OIL_TWT_RADIUS_MS, oil_max_penalty=MAX_OIL_4D_PENALTY,
                   outside_cap=OUTSIDE_CLOSURE_RFI_CAP, contour_ms=CONTOUR_INTERVAL_MS,
                   relief_ms=MIN_CLOSURE_RELIEF_MS, area_m2=MIN_CLOSURE_AREA_M2,
                   closure_smooth=CLOSURE_SMOOTH_BINS, platform_mode=PLATFORM_MODE,
                   platform_index=PLATFORM_INDEX, platform_radius=PLATFORM_RADIUS_M):
    d=pd.read_csv(attr_files[name])
    features={
        'base_rms':percentile_score(d.RMS_Baseline,True),
        'monitor_rms':percentile_score(d.RMS_Monitor,True),
        'base_sweet':percentile_score(d.Sweetness_Baseline,True),
        'monitor_sweet':percentile_score(d.Sweetness_Monitor,True),
        'ndrms':percentile_score(np.abs(d.NdRMS_pct),False),
        'nrms':percentile_score(d.NRMS,False),
        'sweet_change':percentile_score(np.abs(d.Sweetness_Difference),False)}
    keys=list(WEIGHTS)
    w=np.asarray([float(weights[k]) for k in keys])
    if np.any(w<0) or not np.isfinite(w).all() or w.sum()<=0:raise ValueError('Invalid RFI weights')
    w=w/w.sum()
    v=np.stack([features[k] for k in keys],axis=1)
    present=np.isfinite(v)
    effective=(present*w).sum(axis=1)
    base=np.full(len(d),np.nan)
    good=(present.sum(axis=1)>=3)&(effective>0)
    base[good]=100*(np.where(present,v,0)*w).sum(axis=1)[good]/effective[good]
    # 4D response risk (large NdRMS/NRMS/sweetness change -> high risk).
    change=np.stack([1-features[k] for k in ['ndrms','nrms','sweet_change']],axis=1)
    cw=np.array([.70,.20,.10]);mask=np.isfinite(change);den=(mask*cw).sum(axis=1)
    response=np.full(len(d),np.nan);ok=den>0
    response[ok]=(np.where(mask,change,0)*cw).sum(axis=1)[ok]/den[ok]
    oil_support=_scaled_oil_support(d,phase4_oil,oil_xy_radius,oil_twt_radius)
    penalty=np.where(np.isfinite(response),oil_max_penalty*oil_support*response,np.nan)
    closure,paths=_closure_mask(d,interval_ms=contour_ms,min_relief_ms=relief_ms,
                                 min_area_m2=area_m2,smoothing_bins=closure_smooth)
    final=np.where(good&np.isfinite(penalty),np.clip(base-penalty,0,100),np.nan)
    final=np.where(np.isfinite(final)&~closure,np.minimum(final,outside_cap),final)
    scope=np.ones(len(d),dtype=bool)
    if platform_mode!='unconstrained':
        if platforms.empty:raise FileNotFoundError(f'Platform scope requires {PLATFORM_CSV}')
        if platform_radius<=0:raise ValueError('Platform radius must be >0')
        p=platforms if platform_mode=='all_platforms' else platforms.iloc[[int(platform_index)]]
        if platform_mode not in ('all_platforms','single_platform'):raise ValueError('Unknown platform mode')
        dist=cKDTree(p[['X','Y']].to_numpy(float)).query(d[['X','Y']].to_numpy(float))[0]
        scope=dist<=platform_radius
    d['RFI_Base_Weighted']=base;d['Oil_Proximity_Support']=oil_support
    d['FourD_Risk']=response;d['Oil_4D_Penalty']=penalty
    d['Verified_Structural_Closure']=closure
    d['Within_Selected_Platform_Scope']=scope
    d['RFI_Final']=np.where(scope,final,np.nan)
    return d,paths

# Reuse gridded TWT surface across interactive redraws for speed.
_TWT_CONTOUR_CACHE = {}

def horizon_twt_contour_grid(name, d, grid_n=190, max_samples=35000):
    key=(name,int(grid_n),int(max_samples))
    if key in _TWT_CONTOUR_CACHE:
        return _TWT_CONTOUR_CACHE[key]
    xy=d[['X','Y']].to_numpy(dtype=float)
    z=pd.to_numeric(d['TWT'],errors='coerce').to_numpy(dtype=float)
    valid=np.isfinite(xy).all(axis=1)&np.isfinite(z)
    idx=np.flatnonzero(valid)
    if len(idx)<30:
        raise ValueError('Insufficient valid horizon XY/TWT for contouring')
    if len(idx)>max_samples:
        idx=idx[np.linspace(0,len(idx)-1,max_samples).astype(int)]
    x,y,z=xy[idx,0],xy[idx,1],z[idx]
    assert len(x)==len(y)==len(z), "Contour X/Y/TWT alignment failure"
    xx=np.linspace(float(x.min()),float(x.max()),grid_n)
    yy=np.linspace(float(y.min()),float(y.max()),grid_n)
    gx,gy=np.meshgrid(xx,yy)
    try:
        zz=griddata(np.column_stack([x,y]),z,(gx,gy),method='linear')
    except Exception as exc:
        raise ValueError(f'{name}: structural contour interpolation failed: {exc}') from exc
    # Prevent contours extending through unsupported survey gaps.
    dist=cKDTree(np.column_stack([x,y])).query(
        np.column_stack([gx.ravel(),gy.ravel()]),workers=-1
    )[0].reshape(gx.shape)
    # Estimate typical point spacing using source coordinates.
    tree=cKDTree(np.column_stack([x,y]))
    near=tree.query(np.column_stack([x,y]),k=2,workers=-1)[0][:,1]
    spacing=float(np.median(near[np.isfinite(near)&(near>0)])) if np.any((near>0)&np.isfinite(near)) else 100.
    zz[dist>max(4*spacing,2*max(np.diff(xx).mean(),np.diff(yy).mean()))]=np.nan
    _TWT_CONTOUR_CACHE[key]=(gx,gy,zz)
    return gx,gy,zz

def show_rfi_3d(name,save=False,show_contours=True,contour_interval_ms=10.,
                contour_labels=False,**kwargs):
    d,paths=compute_rfi_3d(name,**kwargs)
    rng=np.random.default_rng(RANDOM_SEED)
    eligible=np.flatnonzero(np.isfinite(d.X)&np.isfinite(d.Y)&d.Within_Selected_Platform_Scope.to_numpy(bool))
    ids=rng.choice(eligible,MAX_PLOT_POINTS,replace=False) if len(eligible)>MAX_PLOT_POINTS else eligible
    panels=[
        ('RFI_Base_Weighted','Weighted seismic RFI','RdYlGn',0,100),
        ('Oil_4D_Penalty','Oil proximity × 4D penalty','magma',0,None),
        ('NdRMS_pct','Signed NdRMS (%)','RdBu_r',None,None),
        ('dRMS','dRMS (Baseline − Monitor)','RdBu_r',None,None),
        ('NRMS','NRMS (%)','viridis',0,None),
        ('Sweetness_Difference','Sweetness change (Monitor − Baseline)','RdBu_r',None,None),
        ('Verified_Structural_Closure','Structural closure (screening)','Greys',0,1),
        ('RFI_Final','Final closure-constrained RFI','RdYlGn',0,100)
    ]
    if show_contours:
        if not np.isfinite(contour_interval_ms) or contour_interval_ms<=0:
            raise ValueError('Contour interval must be > 0 ms')
        gx,gy,gz=horizon_twt_contour_grid(name,d)
        finite=gz[np.isfinite(gz)]
        if len(finite):
            low=np.ceil(float(finite.min())/contour_interval_ms)*contour_interval_ms
            high=np.floor(float(finite.max())/contour_interval_ms)*contour_interval_ms
            levels=np.arange(low,high+contour_interval_ms*.5,contour_interval_ms)
            if len(levels)>100:
                raise ValueError('Too many contours (>100). Increase contour interval.')
        else:
            levels=np.array([])
    else:
        levels=np.array([])
    with plt.rc_context({'font.size':PLOT_FONT_SIZE,
                         'axes.titlesize':PLOT_FONT_SIZE+1,
                         'axes.labelsize':PLOT_FONT_SIZE}):
        fig,axes=plt.subplots(2,4,figsize=(25,13),constrained_layout=True)
        for ax,(col,title,cmap,vmin,vmax) in zip(axes.flat,panels):
            values=pd.to_numeric(d[col],errors='coerce').to_numpy(dtype=float)
            good=ids[np.isfinite(values[ids])]
            if not len(good):
                ax.set_title(title+' — no data')
                continue
            if col in ('NdRMS_pct','dRMS','Sweetness_Difference'):
                lim=max(1e-8,float(np.nanpercentile(np.abs(values[good]),98)))
                vmin,vmax=-lim,lim
            elif vmax is None:
                vmax=max(1e-8,float(np.nanpercentile(values[good],98)))
            artist=ax.scatter(d.X.iloc[good],d.Y.iloc[good],c=values[good],s=1,
                              cmap=cmap,vmin=vmin,vmax=vmax,rasterized=True)
            fig.colorbar(artist,ax=ax,shrink=.75,pad=.025)
            if len(levels) and np.isfinite(gz).any():
                cs=ax.contour(gx,gy,np.ma.masked_invalid(gz),levels=levels,
                              colors='black',linewidths=.55,alpha=.65,zorder=7)
                if contour_labels:
                    ax.clabel(cs,inline=True,fontsize=max(7,PLOT_FONT_SIZE-5),
                              fmt=lambda v:f'{v:g}')
            ax.set_title(title,pad=10)
            ax.set_xlabel('X (m)')
            ax.set_ylabel('Y (m)')
            ax.set_aspect('equal',adjustable='datalim')
            ax.tick_params(labelsize=max(8,PLOT_FONT_SIZE-2))
            if not platforms.empty:
                ax.scatter(platforms.X,platforms.Y,marker='^',s=45,
                           facecolor='white',edgecolor='black',zorder=10)
        fig.suptitle(f'Phase 6 Part 3D — {name} | Direct SEG-Y ±{HALF_WINDOW_MS:g} ms',
                     fontsize=PLOT_FONT_SIZE+4)
        if save:
            safe=re.sub(r'[^A-Za-z0-9_.-]+','_',name)
            d.to_csv(RFI_OUT/(safe+'_RFI.csv'),index=False,float_format='%.12g')
            fig.savefig(RFI_OUT/(safe+'_RFI.png'),dpi=PLOT_DPI,bbox_inches='tight')
            print('Saved:',RFI_OUT/(safe+'_RFI.csv'))
        plt.show()
        plt.close(fig)
    print('Oil samples:',len(phase4_oil),'| structural-high polygons:',len(paths),
          '| closure bins:',int(d.Verified_Structural_Closure.sum()),
          '| finite RFI:',int(d.RFI_Final.notna().sum()))
    return d

# Wide, readable controls; labels are above inputs to prevent truncation.
names=list(attr_files)
def labeled_control(label, widget, width='255px'):
    widget.description=''
    widget.layout=widgets.Layout(width='100%')
    return widgets.VBox([
        widgets.HTML('<div style="font-size:13px;line-height:1.35;font-weight:600;white-space:normal;">'+label+'</div>'),
        widget
    ],layout=widgets.Layout(width=width,margin='0 0 8px 0'))

horizon_selector=widgets.Dropdown(options=names,value='I-25' if 'I-25' in names else names[0])
scope_selector=widgets.Dropdown(options=[
    ('Full survey (no platform constraint)','unconstrained'),
    ('Within radius of any platform','all_platforms'),
    ('Within radius of selected platform','single_platform')
],value='unconstrained')
platform_options=[(str(row.Name),i) for i,row in platforms.iterrows()]
platform_selector=widgets.Dropdown(options=platform_options or [('No platform data',0)])
platform_selector.disabled=platforms.empty
weights_ui={k:widgets.FloatText(value=v) for k,v in WEIGHTS.items()}
oilxy_ui=widgets.FloatText(value=OIL_XY_RADIUS_M)
oilt_ui=widgets.FloatText(value=OIL_TWT_RADIUS_MS)
penalty_ui=widgets.FloatText(value=MAX_OIL_4D_PENALTY)
cap_ui=widgets.FloatText(value=OUTSIDE_CLOSURE_RFI_CAP)
radius_ui=widgets.FloatText(value=PLATFORM_RADIUS_M)
contour_ui=widgets.FloatText(value=CONTOUR_INTERVAL_MS)
closure_interval_ui=widgets.FloatText(value=CONTOUR_INTERVAL_MS)
relief_ui=widgets.FloatText(value=MIN_CLOSURE_RELIEF_MS)
area_ui=widgets.FloatText(value=MIN_CLOSURE_AREA_M2)
smooth_ui=widgets.FloatText(value=CLOSURE_SMOOTH_BINS)
contour_toggle=widgets.Checkbox(value=True,description='Show structural TWT contours',indent=False)
contour_label_toggle=widgets.Checkbox(value=False,description='Label contours (ms)',indent=False)
run_button=widgets.Button(description='Update RFI maps',button_style='primary',layout=widgets.Layout(width='190px'))
save_button=widgets.Button(description='Export RFI CSV + PNG',button_style='success',layout=widgets.Layout(width='215px'))
output=widgets.Output()

def _run(save=False):
    with output:
        clear_output(wait=True)
        try:
            show_rfi_3d(
                horizon_selector.value,save=save,
                show_contours=contour_toggle.value,
                contour_interval_ms=contour_ui.value,
                contour_labels=contour_label_toggle.value,
                weights={k:v.value for k,v in weights_ui.items()},
                oil_xy_radius=oilxy_ui.value,
                oil_twt_radius=oilt_ui.value,
                oil_max_penalty=penalty_ui.value,
                outside_cap=cap_ui.value,
                contour_ms=closure_interval_ui.value,
                relief_ms=relief_ui.value,
                area_m2=area_ui.value,
                closure_smooth=smooth_ui.value,
                platform_mode=scope_selector.value,
                platform_index=platform_selector.value,
                platform_radius=radius_ui.value
            )
        except Exception as exc:
            print(type(exc).__name__+':',exc)
            raise

run_button.on_click(lambda _: _run(False))
save_button.on_click(lambda _: _run(True))
scope_selector.observe(
    lambda change: setattr(platform_selector,'disabled',
        platforms.empty or change['new']!='single_platform'),
    names='value'
)
platform_selector.disabled=True
weight_labels={
    'base_rms':'Baseline RMS weight',
    'monitor_rms':'Monitor RMS weight',
    'base_sweet':'Baseline sweetness weight',
    'monitor_sweet':'Monitor sweetness weight',
    'ndrms':'Absolute signed NdRMS weight',
    'nrms':'NRMS weight',
    'sweet_change':'Absolute sweetness change weight'
}
layout=widgets.Layout(grid_template_columns='repeat(3, minmax(260px, 1fr))',
                      grid_gap='10px 18px',width='100%')
display(widgets.VBox([
    widgets.HTML('<h3 style="margin-bottom:8px;">Phase 6 Part 3D — Horizon RFI viewer</h3>'),
    widgets.GridBox([
        labeled_control('Selected horizon',horizon_selector),
        labeled_control('Platform scope',scope_selector),
        labeled_control('Selected platform',platform_selector)
    ],layout=layout),
    widgets.HTML('<h4 style="margin:12px 0 4px;">Seismic attribute weights (relative, normalized automatically)</h4>'),
    widgets.GridBox([labeled_control(weight_labels[k],v) for k,v in weights_ui.items()],layout=layout),
    widgets.HTML('<h4 style="margin:12px 0 4px;">Oil proximity, structural closure and platform controls</h4>'),
    widgets.GridBox([
        labeled_control('Oil lateral search radius (m)',oilxy_ui),
        labeled_control('Oil vertical TWT radius (ms)',oilt_ui),
        labeled_control('Maximum oil × 4D penalty (RFI points)',penalty_ui),
        labeled_control('Outside-closure RFI cap (0–100)',cap_ui),
        labeled_control('Platform search radius (m)',radius_ui),
        labeled_control('Display TWT contour interval (ms)',contour_ui),
        labeled_control('Closure detection interval (ms)',closure_interval_ui),
        labeled_control('Minimum closure relief (ms)',relief_ui),
        labeled_control('Minimum closure area (m²)',area_ui),
        labeled_control('Closure smoothing (grid bins)',smooth_ui)
    ],layout=layout),
    widgets.HBox([contour_toggle,contour_label_toggle],
                 layout=widgets.Layout(gap='30px',flex_flow='row wrap')),
    widgets.HBox([run_button,save_button],layout=widgets.Layout(gap='12px')),
    output
],layout=widgets.Layout(width='100%')))
print('Cell 7 completed: select a horizon and click Update RFI maps')


Cell 7 completed: select a horizon and click Update RFI maps


### Important interpretation and QC limitations

- `NdRMS` and `NRMS` are computed from the same ±10 ms paired seismic samples. Sweetness comes from 3B and has a different temporal support; no attempt is made to force it to match RMS.
- Structural closure is a **contour-based screening proxy**, not a validated sealed trap. The cap outside identified structural highs is intentionally conservative.
- Oil proximity requires usable Phase 4 class-2 samples and actual checkshot/deviation-to-XY trajectories; missing deviation is treated as vertical. Wells without valid checkshot/XY are excluded with printed QC, not fabricated.
- Scoring is percentile-normalized **per horizon**; absolute RFI values are not directly calibrated across horizons or to saturation/remaining volumes. The user should review geological meaning and production data before interpreting as fluid potential.
- Seismic time axis and horizon TWT must both be milliseconds; SEG-Y trace order is validated using inline/crossline keys.
